In [1]:
#=================================================
# Most of the time we use Deep Lake for VectorDB
#=================================================
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

In [5]:
# 1. Load environment variables (from local dir and project root)
# current_dir = Path(__file__).parent
# load_dotenv(current_dir / ".env")
load_dotenv()

True

In [6]:
activeloop_token = (
    os.getenv("ACTIVELOOP_TOKEN")
    or os.getenv("ACTIVELOOP_API_KEY")
    or os.getenv("DEEPLAKE_API_KEY")
    or os.getenv("DEEPLAKE_API_TOKEN")
)

In [7]:
if activeloop_token:
    os.environ["ACTIVELOOP_TOKEN"] = activeloop_token


In [8]:
# 2. Compatibility shim: LangChain 1.x moved retrievers to langchain-classic,
# but langchain-deeplake expects langchain.retrievers
import langchain
import langchain_classic.retrievers
langchain.retrievers = langchain_classic.retrievers
import langchain_classic.retrievers.self_query.base
sys.modules["langchain.retrievers"] = langchain_classic.retrievers
sys.modules["langchain.retrievers.self_query"] = langchain_classic.retrievers.self_query
sys.modules["langchain.retrievers.self_query.base"] = langchain_classic.retrievers.self_query.base


In [9]:
from langchain_deeplake import DeeplakeVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Use a modern chat model instead of the legacy completion model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


In [10]:
# Create documents
texts = [
    "Napoleon Bonaparte was born in 15 August 1769",
    "Louis XIV was born in 5 September 1638"
]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.create_documents(texts)


In [11]:
# Create Deep Lake dataset
# Activeloop username/org slug (must NOT contain spaces or apostrophes)
my_activeloop_org_id = "sheshpalsingh2024"
my_activeloop_dataset_name = "langchain_course_from_zero_to_hero"
dataset_path = f"hub://{my_activeloop_org_id}/{my_activeloop_dataset_name}"


In [12]:
# Note: You can also use a local path if working offline:
# dataset_path = "./data/deeplake_db"

db = DeeplakeVectorStore(
    dataset_path=dataset_path,
    embedding_function=embeddings,
    token=activeloop_token,
    overwrite=True,
)

In [13]:
db.add_documents(docs)

# Build a retriever (modern vector store API)
retriever = db.as_retriever(search_kwargs={"k": 2})


In [14]:
# Modern RAG: a simple function instead of RetrievalQA
def rag_query(question: str) -> str:
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = (
        "Use the context below to answer the question. "
        "If you don't know, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    return llm.invoke(prompt).content


In [15]:
# Run it
print(rag_query("When was Napoleon Bonaparte born?"))

Napoleon Bonaparte was born on 15 August 1769.


In [17]:
## Add Dataset in DB
# load the existing Deep Lake dataset and specify the embedding function
db = DeeplakeVectorStore(dataset_path=dataset_path, embedding_function=embeddings)

# create new documents
texts = [
    "Lady Gaga was born in 28 March 1986",
    "Michael Jeffrey Jordan was born in 17 February 1963"
]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.create_documents(texts)

# add documents to our Deep Lake dataset
db.add_documents(docs)

['18eb6747-16a4-43b7-b3bb-1040b88cdd2b',
 '380a2594-269c-4939-a1b2-36f454253b46']

In [19]:
from langchain_classic.chains import RetrievalQA
from langchain_classic.agents import initialize_agent, Tool, AgentType
from langchain_openai import ChatOpenAI

# Use modern ChatOpenAI (gpt-3.5-turbo-instruct is deprecated)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# create a retriever from the db
retrieval_qa = RetrievalQA.from_chain_type(
	llm=llm, chain_type="stuff", retriever=db.as_retriever()
)

# instantiate a tool that uses the retriever
tools = [
    Tool(
        name="Retrieval QA System",
        func=retrieval_qa.run,
        description="Useful for answering questions."
    ),
]

# create an agent that uses the tool
agent = initialize_agent(
	tools,
	llm,
	agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
	verbose=True
)


/tmp/ipykernel_33902/4059939794.py:23: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(


In [20]:
# Run query with the agent
response = agent.run("When was Michael Jordan born?")
print(response)


/tmp/ipykernel_33902/3649392020.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = agent.run("When was Michael Jordan born?")




> Entering new AgentExecutor chain...
I need to find the birth date of Michael Jordan. This information is likely available in general knowledge databases or sports history resources. 
Action: Retrieval QA System
Action Input: "When was Michael Jordan born?"
Observation: Michael Jordan was born on 17 February 1963.
Thought:I now know the final answer.  
Final Answer: Michael Jordan was born on February 17, 1963.

> Finished chain.
Michael Jordan was born on February 17, 1963.
